## GigaAM Transcribation for all files

#### Imports and preparations

In [ ]:
import os
from pathlib import Path
from pydub import AudioSegment
from math import ceil
from tqdm import tqdm
import json
import gigaam
import time
import torch

# Папки
ANNOT_DIR = Path("annot")
SEGMENT_DIR = Path("segments")
SEGMENT_DIR.mkdir(exist_ok=True)

# Параметры сегментации
MIN_SEG_LEN = 18_000  # 18 сек
MAX_SEG_LEN = 30_000  # safety limit для Whisper
SAMPLE_RATE = 16_000


#### Script for transcribations

In [ ]:
def split_audio(audio: AudioSegment, basename: str, output_dir: Path):
    duration_ms = len(audio)
    segments = []

    if duration_ms <= MIN_SEG_LEN:
        out_path = output_dir / f"{basename}_seg0.wav"
        audio.export(out_path, format="wav")
        segments.append((out_path, len(audio) / 1000))
        return segments

    num_segments = ceil(duration_ms / MIN_SEG_LEN)
    ideal_len = duration_ms // num_segments

    while ideal_len > MAX_SEG_LEN and num_segments < duration_ms // 1000:
        num_segments += 1
        ideal_len = duration_ms // num_segments

    for i in range(num_segments):
        start = i * ideal_len
        end = min((i + 1) * ideal_len, duration_ms)
        segment = audio[start:end]
        out_path = output_dir / f"{basename}_seg{i}.wav"
        segment.export(out_path, format="wav")
        segments.append((out_path, len(segment) / 1000))
    return segments

In [ ]:
all_segments = []

for wav_path in tqdm(ANNOT_DIR.glob("*.wav")):
    audio = AudioSegment.from_file(wav_path).set_frame_rate(SAMPLE_RATE).set_channels(1)
    basename = wav_path.stem
    segments = split_audio(audio, basename, SEGMENT_DIR)
    all_segments.extend(segments)

print(f"Нарезано сегментов: {len(all_segments)}")


In [ ]:
SEGMENT_DIR = Path("segments")
TXT_DIR = Path("transcripts")
TXT_DIR.mkdir(exist_ok=True)

manifest = []

asr_model = gigaam.load_model("v2_rnnt")
print("GigaAM загружена")


In [ ]:
# os.environ["HF_TOKEN"] = "YOUR_TOKEN"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
asr_model.to(device)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: False
Device: CPU only


In [ ]:
LOG_EVERY = 500

for i, wav_path in enumerate(tqdm(sorted(SEGMENT_DIR.glob("*.wav")))):
    txt_path = TXT_DIR / (wav_path.stem + ".txt")

    try:
        if txt_path.exists():
            with open(txt_path, "r", encoding="utf-8") as f:
                text = f.read().strip()
        else:
            segments = asr_model.transcribe_longform(str(wav_path))
            text = " ".join([s["transcription"].strip() for s in segments if s["transcription"].strip()])
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(text)

        duration_sec = float(os.popen(
            f"ffprobe -i \"{wav_path}\" -show_entries format=duration -v quiet -of csv=\"p=0\""
        ).read().strip())

        manifest.append({
            "audio_filepath": str(wav_path.as_posix()),
            "text": text,
            "duration": duration_sec
        })

    except Exception as e:
        print(f"Ошибка на {wav_path.name}: {e}")
        continue

    if (i + 1) % LOG_EVERY == 0:
        print(f"✓ Обработано: {i + 1} / {len(list(SEGMENT_DIR.glob('*.wav')))}")

  1%|          | 504/79216 [00:20<1:06:58, 19.59it/s]

✓ Обработано: 500 / 79216


  1%|▏         | 1002/79216 [00:40<1:40:57, 12.91it/s]

✓ Обработано: 1000 / 79216


  2%|▏         | 1503/79216 [01:00<1:26:43, 14.93it/s]

...

 99%|█████████▉| 78500/79216 [11:04:31<18:42,  1.57s/it]

✓ Обработано: 78500 / 79216


100%|█████████▉| 79000/79216 [11:17:25<06:22,  1.77s/it]

✓ Обработано: 79000 / 79216


100%|██████████| 79216/79216 [11:23:00<00:00,  1.93it/s]


#### Saving transcribations into manifest

In [ ]:
with open("manifest.jsonl", "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"manifest.jsonl создан: {len(manifest)} сегментов")


In [6]:
from collections import Counter
names = [p.name for p in SEGMENT_DIR.glob("*.wav")]
dupes = [item for item, count in Counter(names).items() if count > 1]
print("Повторы:", dupes)

Повторы: []


In [7]:
segment_paths = list(SEGMENT_DIR.glob("*.wav"))
print(f"Найдено файлов: {len(segment_paths)}")
print(f"Уникальных имён: {len(set(p.stem for p in segment_paths))}")


Найдено файлов: 79216
Уникальных имён: 79216


In [ ]:
MANIFEST_PATH = Path("manifest.jsonl")
manifest = []

all_wavs = sorted(SEGMENT_DIR.glob("*.wav"))

for wav_path in tqdm(all_wavs):
    txt_path = TXT_DIR / (wav_path.stem + ".txt")
    if not txt_path.exists():
        continue

    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
        if not text:
            continue  

    try:
        duration_sec = float(os.popen(
            f"ffprobe -i \"{wav_path}\" -show_entries format=duration -v quiet -of csv=\"p=0\""
        ).read().strip())
    except Exception as e:
        print(f"Ошибка чтения длительности: {wav_path.name} — {e}")
        continue

    manifest.append({
        "audio_filepath": str(wav_path.as_posix()),
        "text": text,
        "duration": duration_sec
    })

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Готово: manifest.jsonl создан ({len(manifest)} сегментов)")